In [1]:
#Import Libs
import numpy as np
import binascii
import re

In [2]:
#Hopfield Activation Function (Bipolar)
def activation_function(x):
    if x < 0:
        return -1
    return 1

In [3]:
#Information Retrieval Matrix
class Matrix:

    @staticmethod
    def matrix_vector_multiplication(matrix, vector):
        return matrix.dot(vector)

    @staticmethod
    def clear_diagonal(matrix):
        np.fill_diagonal(matrix, 0)
        return matrix

    @staticmethod
    def outer_product(pattern):
        return np.outer(pattern, pattern)

    @staticmethod
    def add_matrices(matrix1, matrix2):
        return matrix1 + matrix2

In [4]:
#Hopfield Structure, the Train and Retrieval Steps
class HopfieldNetwork:

    def __init__(self, dimension):
        self.weight_matrix = np.zeros((dimension, dimension))

    def train(self, pattern):

        pattern_bipolar = HopfieldNetwork.transform(pattern)

        pattern_weight_matrix = Matrix.outer_product(pattern_bipolar)

        pattern_weight_matrix = Matrix.clear_diagonal(pattern_weight_matrix)

        self.weight_matrix = Matrix.add_matrices(self.weight_matrix, pattern_weight_matrix)

    def recall(self, pattern):

        pattern_bipolar = HopfieldNetwork.transform(pattern)

        result = Matrix.matrix_vector_multiplication(self.weight_matrix, pattern_bipolar)

        result = np.array([activation_function(x) for x in result])

        result = HopfieldNetwork.re_transform(result)

        return(result)

    @staticmethod
    def transform(pattern):
        return np.where(pattern == 0, -1, pattern)

    @staticmethod
    def re_transform(pattern):
        return np.where(pattern == -1, 0, pattern)

In [5]:
#HNN with 128 neurons, one to each character in the Hash
network = HopfieldNetwork(128)

<b> Function to Conversions Tasks

In [6]:
#Text to Binary
def text_to_bits(text, encoding='utf-8', errors='surrogatepass'):
    bits = bin(int(binascii.hexlify(text.encode(encoding, errors)), 16))[2:]
    return bits.zfill(8 * ((len(bits) + 7) // 8))

def text_from_bits(bits, encoding='utf-8', errors='surrogatepass'):
    n = int(bits, 2)
    return int2bytes(n).decode(encoding, errors)

def int2bytes(i):
    hex_string = '%x' % i
    n = len(hex_string)
    return binascii.unhexlify(hex_string.zfill(n + (n & 1)))

In [7]:
#Hash Text to Binary and Split into a List
def conv_ascii(string):
    string = str(text_to_bits(string))
    string = np.array(list(map(int, string)))
    return string

In [8]:
#ASCII to Text
def conv_string(string):

    x = ""
    for i in range(len(string)):
        x = x + str(string[i])

    n = int(x, 2)
    x = binascii.unhexlify('%x' % n)

    x = str(x)
    x = x.split("'")
    x = x[1]

    return(x)

<b> Experiments Step - List of Original Hash Values to Train HNN

In [9]:
#All the Original values
hash1 = "7670795b33135a38" #Lenna Database
hash2 = "d3d85833daeab5a9" #Washington Database
hash3 = "e6ce8e991c149694" #Palace Database
hash4 = "402416531b191a1f" #Mountain Database
hash5 = "4659d98bcbcb9639" #Park Database

In [10]:
#Conversion of the Hash Values
h1 = conv_ascii(hash1)
h2 = conv_ascii(hash2)
h3 = conv_ascii(hash3)
h4 = conv_ascii(hash4)
h5 = conv_ascii(hash5)

In [11]:
#HNN Train
network.train(h1)
network.train(h2)
network.train(h3)
network.train(h4)
network.train(h5)

**Select the Original Hash Value for Evaluation**

In [12]:
#Select the Original Value for Evaluation
print("Select an option for the original hash:")
print("1: Lenna Database")
print("2: Washington Database")
print("3: Palace Database")
print("4: Mountain Database")
print("5: Park Database")

choice = input("Enter the number of your choice: ")

original = ""

if choice == '1':
    original = hash1
elif choice == '2':
    original = hash2
elif choice == '3':
    original = hash3
elif choice == '4':
    original = hash4
elif choice == '5':
    original = hash5
else:
    print("Invalid option. Please choose a number from 1 to 5.")

if original:
    print(f"You selected: {original}")
else:
    print("No hash was selected due to an invalid choice.")

Select an option for the original hash:
1: Lenna Database
2: Washington Database
3: Palace Database
4: Mountain Database
5: Park Database
Enter the number of your choice: 1
You selected: 7670795b33135a38


<b> Input of Noise Information

In [13]:
#Input Hash Altered Value
a_hash = str(input("Hash Value: "))

Hash Value: 6470795b33135a38


In [14]:
#Conversion of Hash Altered Value to ASCII
ascii_a_hash = conv_ascii(a_hash)
print(ascii_a_hash)

[0 0 1 1 0 1 1 0 0 0 1 1 0 1 0 0 0 0 1 1 0 1 1 1 0 0 1 1 0 0 0 0 0 0 1 1 0
 1 1 1 0 0 1 1 1 0 0 1 0 0 1 1 0 1 0 1 0 1 1 0 0 0 1 0 0 0 1 1 0 0 1 1 0 0
 1 1 0 0 1 1 0 0 1 1 0 0 0 1 0 0 1 1 0 0 1 1 0 0 1 1 0 1 0 1 0 1 1 0 0 0 0
 1 0 0 1 1 0 0 1 1 0 0 1 1 1 0 0 0]


In [15]:
#Retrive Information with HNN
hr = network.recall(ascii_a_hash);
print(hr)

[0 0 1 1 0 1 1 1 0 0 1 1 0 1 1 0 0 0 1 1 0 1 1 1 0 0 1 1 0 0 0 0 0 0 1 1 0
 1 1 1 0 0 1 1 1 0 0 1 0 0 1 1 0 1 0 1 0 1 1 0 0 0 1 0 0 0 1 1 0 0 1 1 0 0
 1 1 0 0 1 1 0 0 1 1 0 0 0 1 0 0 1 1 0 0 1 1 0 0 1 1 0 1 0 1 0 1 1 0 0 0 0
 1 0 0 1 1 0 0 1 1 0 0 1 1 1 0 0 0]


In [16]:
#Conversion of Hash Altered Value into a String
hr = str(conv_string(hr))
print(hr)

7670795b33135a38


<b> HNN Validation with Hamming

In [17]:
#Show Values
print("Org_Hash: " + str(original))
print("Alt_Hash: " + str(a_hash))
print("Ret_Hash: " + str(hr))

Org_Hash: 7670795b33135a38
Alt_Hash: 6470795b33135a38
Ret_Hash: 7670795b33135a38


In [18]:
#Hamming Distance Function
def hamming(hash1,hash2):
    L = len(hash1)
    distance = 0

    for i in range(L):
        if hash1[i] != hash2[i]:
            distance += 1

    return(str(distance))

In [19]:
#Hamming Distance
hash_dist1 = hamming(original,a_hash)
hash_dist2 = hamming(original,hr)

print("Hamming Distance Between Original and Altered: " + str(hash_dist1))
print("Hamming Distance Between Original and Retrived: " + str(hash_dist2))

Hamming Distance Between Original and Altered: 2
Hamming Distance Between Original and Retrived: 0
